<a href="https://colab.research.google.com/github/alianas-dev/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alianas-dev/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup: DuckDB over the warehouse, HF token from Colab Secrets, sklearn for modeling.
%pip install -q duckdb scikit-learn

import duckdb, os
import pandas as pd
import numpy as np

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        from getpass import getpass
        HF_TOKEN = getpass("Enter your HF_TOKEN (not saved in this notebook): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
FACT_FEB    = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet')"
FACT_MAR    = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
FEB_END     = "DATE '2026-02-28'"

# Genuine past -> future design: ALL features come from February 2026 (the past window).
# The label comes from what actually happened in March 2026 (the future window) — a real
# forward-looking prediction, not a same-month proxy. Both months are mid-panel; June 2026
# (the sealed _sample test month) is never touched anywhere in this notebook.
print("DuckDB ready. Features: Feb 2026 (past). Label: Mar 2026 (future, unseen at decision time).")

DuckDB ready. Features: Feb 2026 (past). Label: Mar 2026 (future, unseen at decision time).


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Question shape.** My lane (Refresh / Content Opportunity Scoring) is a ranking question — "which pages first?" — with an observed forward label I can build honestly: did a page's search impressions actually fall next month? Per the method-choice table, that combination (ranking + an observed label) means starting with a classifier's *probability*, evaluated at precision@K, not a bare classification accuracy.

**Methods used.**
- **Logistic Regression** — the readable starting point. A ranking built from a linear combination of a handful of interpretable signals (staleness, CTR gap, position, volume) is something a non-engineer can be walked through, the same spirit as the Week-4 rule, just with fitted weights instead of hand-picked thresholds.
- **Random Forest** — tests whether the signals interact in ways a straight line can't capture (Week-3's finding was that no single signal correlates strongly alone, and a hand-written AND-rule actually underperformed the base rate — that's exactly the kind of interaction a tree ensemble can pick up that a linear model or an if-statement can't).
- **Permutation importance** on the stronger model, to say *what it leans on* — not just that it works.

Simplicity is treated as a feature here, not a default to abandon: Random Forest only earns its place in Section 3's table if it actually beats Logistic Regression by a real margin, on the same split and metric. If it doesn't, the honest conclusion is to prefer the simpler model — that judgment happens in Section 3, from the numbers, not decided in advance here.

**Baseline.** The Week-4 rule (`stale AND ctr_below_position_band_expectation AND visible`), recomputed here on the *same* February feature window so the comparison in Section 3 is apples-to-apples — same rows, same split, same metric, computed in this same notebook run.

In [2]:
# Build the feature (Feb) + label (Mar) frame in one query, and recompute the Week-4
# rule's score using ONLY February data (its own position-band lookup, fit on Feb, not Mar).
frame = con.sql(f"""
    WITH feb_agg AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(gsc_impressions) AS impressions_feb,
            SUM(gsc_clicks) AS clicks_feb,
            AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_feb,
            COUNT(*) AS days_with_data_feb,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS sessions_feb
        FROM {FACT_FEB}
        GROUP BY client_hash_id, content_hash_id
    ),
    mar_agg AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(gsc_impressions) AS impressions_mar
        FROM {FACT_MAR}
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        f.*,
        c.content_type,
        c.content_updated_date,
        c.content_created_date,
        DATE_DIFF('day', COALESCE(c.content_updated_date, c.content_created_date), {FEB_END}) AS days_since_update,
        CASE WHEN f.impressions_feb > 0 THEN f.clicks_feb * 1.0 / f.impressions_feb END AS ctr_feb,
        COALESCE(m.impressions_mar, 0) AS impressions_mar,
        -- a page with zero March rows (dropped out of the panel entirely) still counts as
        -- a maximal decline here; Section 4 flags this as a real, separate error pattern.
        CASE WHEN m.content_hash_id IS NULL THEN 1 ELSE 0 END AS disappeared_in_march
    FROM feb_agg f
    JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    LEFT JOIN mar_agg m ON f.content_hash_id = m.content_hash_id AND f.client_hash_id = m.client_hash_id
    WHERE f.impressions_feb > 0
""").df()

# The label: same >20% drop rule used since Week 3, now genuinely forward-looking.
frame["is_declining_next_month"] = (
    (frame["impressions_mar"] - frame["impressions_feb"]) / frame["impressions_feb"] < -0.20
).astype(int)

print(f"Rows (pages with real Feb visibility): {len(frame):,}")
print(f"Label rate (declined by >20% in March): {frame['is_declining_next_month'].mean():.1%}")
print(f"Of those, disappeared entirely from March tracking: {frame['disappeared_in_march'].sum():,}")

# Real-data gotcha: a page can have impressions but no day where GSC recorded a real
# position (every day landed on the 0 = "no data" placeholder), so avg_position_feb
# comes back NULL. That missingness is itself informative — keep it as a flag rather
# than silently imputing it away — then impute the numeric value for the models below.
frame["avg_position_missing"] = frame["avg_position_feb"].isna().astype(int)
frame["days_since_update_missing"] = frame["days_since_update"].isna().astype(int)
print(f"Pages with no valid position data this month: {frame['avg_position_missing'].sum():,}")

# Recompute the Week-4 rule, fit on Feb data only (its own position-band CTR lookup).
frame["position_band"] = pd.cut(
    frame["avg_position_feb"], bins=[0, 3, 10, 20, 50, 100000],
    labels=["1-3", "4-10", "11-20", "21-50", "51+"],
)
band_ctr = frame.groupby("position_band", observed=True)["ctr_feb"].mean()
frame["expected_ctr_for_band"] = frame["position_band"].astype(str).map(band_ctr.to_dict()).astype(float)
frame["ctr_gap"] = (frame["expected_ctr_for_band"] - frame["ctr_feb"]).clip(lower=0).fillna(0)
# .fillna(0) matters here: a page with no position data gets NaN ctr_gap, and
# 0 (underperforming flag) * NaN would silently produce NaN, not 0, in the score below.

stale = (frame["days_since_update"] >= 180).astype(int)
underperforming = (frame["ctr_gap"] > 0).astype(int)
visible = (frame["impressions_feb"] >= 500).astype(int)
frame["baseline_rule_score"] = stale * underperforming * visible * frame["ctr_gap"] * frame["impressions_feb"]

print(f"Week-4 rule flags {int((frame['baseline_rule_score'] > 0).sum()):,} of {len(frame):,} pages this month.")
frame.head(3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows (pages with real Feb visibility): 153,559
Label rate (declined by >20% in March): 30.1%
Of those, disappeared entirely from March tracking: 8,280
Pages with no valid position data this month: 1,603
Week-4 rule flags 26 of 153,559 pages this month.


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,avg_position_feb,days_with_data_feb,sessions_feb,content_type,content_updated_date,content_created_date,...,ctr_feb,impressions_mar,disappeared_in_march,is_declining_next_month,avg_position_missing,days_since_update_missing,position_band,expected_ctr_for_band,ctr_gap,baseline_rule_score
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4270.0,7.0,5.805560,28,0.0,keyword article,2026-07-06,2025-02-28,...,0.001639,6523.0,0,0,0,0,4-10,0.0054,0.003761,0.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,440.0,2.0,4.142846,28,0.0,keyword article,2026-05-18,2025-02-28,...,0.004545,453.0,0,0,0,0,4-10,0.0054,0.000855,0.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5271.0,4.0,7.101588,28,0.0,keyword article,2026-05-20,2025-02-28,...,0.000759,5630.0,0,0,0,0,4-10,0.0054,0.004642,0.0


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by client, not by row.** Multiple content items belong to the same client, and a client's overall site health, niche, or tracking quality can make its pages behave similarly. A row-level random split would let the model see some of a client's pages in training and others in testing — leaking client-level patterns across the split rather than testing whether the model generalizes to *entirely unseen* clients. `GroupShuffleSplit` on `client_hash_id` keeps every client's pages together on one side of the split.

**Already time-aware by construction.** The past→future design in Section 1 (Feb features, March label) means there's no additional time-based split to design here — the leakage boundary is the calendar, not a row cutoff, and it's enforced once at the query level, not per-fold.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(frame, groups=frame["client_hash_id"]))

train_df = frame.iloc[train_idx].reset_index(drop=True)
test_df  = frame.iloc[test_idx].reset_index(drop=True)

train_clients = set(train_df["client_hash_id"])
test_clients  = set(test_df["client_hash_id"])
overlap = train_clients & test_clients

print(f"Train: {len(train_df):,} pages across {len(train_clients)} clients")
print(f"Test:  {len(test_df):,} pages across {len(test_clients)} clients")
print(f"Clients appearing in BOTH train and test (should be 0): {len(overlap)}")
print(f"Label rate — train: {train_df['is_declining_next_month'].mean():.1%}  |  test: {test_df['is_declining_next_month'].mean():.1%}")
assert len(overlap) == 0, "Group split failed — a client leaked across train/test"

Train: 123,574 pages across 34 clients
Test:  29,985 pages across 12 clients
Clients appearing in BOTH train and test (should be 0): 0
Label rate — train: 28.6%  |  test: 36.1%


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

NUMERIC_FEATURES = ["impressions_feb", "ctr_feb", "avg_position_feb", "days_since_update",
                     "sessions_feb", "avg_position_missing", "days_since_update_missing"]
CATEGORICAL_FEATURES = ["content_type"]
FEATURE_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

preprocess = ColumnTransformer([
    # median-impute missing values (e.g. no valid GSC position that month) rather than
    # dropping the row — the *_missing flags above let the model still use the fact
    # that it was missing, instead of that information disappearing into an imputed value.
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])

X_train, y_train = train_df[FEATURE_COLS], train_df["is_declining_next_month"]
X_test, y_test   = test_df[FEATURE_COLS],  test_df["is_declining_next_month"]

logreg = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
]).fit(X_train, y_train)

rf = Pipeline([
    ("prep", preprocess),
    # depth and leaf size kept modest on purpose — this is not a search for the biggest
    # number, it's a check on whether interaction effects beat a straight line at all.
    ("clf", RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=50,
                                    class_weight="balanced", random_state=42)),
]).fit(X_train, y_train)

print("Models fit. Random seed fixed at 42 throughout for reproducibility.")

Models fit. Random seed fixed at 42 throughout for reproducibility.


In [5]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    top_k = np.asarray(labels)[order[:k]]
    return top_k.mean(), int(top_k.sum())

base_rate = y_test.mean()

results = []
for name, scores, has_auc in [
    ("Week-4 rule (baseline)", test_df["baseline_rule_score"].values, False),
    ("Logistic Regression",     logreg.predict_proba(X_test)[:, 1],   True),
    ("Random Forest",           rf.predict_proba(X_test)[:, 1],        True),
]:
    p50, hits50 = precision_at_k(scores, y_test.values, 50)
    p20, hits20 = precision_at_k(scores, y_test.values, 20)
    auc = roc_auc_score(y_test, scores) if has_auc else None
    results.append({
        "method": name, "precision@20": round(p20, 3), "hits@20": hits20,
        "precision@50": round(p50, 3), "hits@50": hits50,
        "AUC": round(auc, 3) if auc is not None else "n/a (step-function score, not a full ranking)",
    })

comparison = pd.DataFrame(results)
print(f"Base rate in test set (random-ranking expectation): {base_rate:.3f}")
print()
comparison

Base rate in test set (random-ranking expectation): 0.361



,method,precision@20,hits@20,precision@50,hits@50,AUC
0,Week-4 rule (baseline),1.0,20,0.86,43,"n/a (step-function score, not a full ranking)"
1,Logistic Regression,0.8,16,0.64,32,0.86
2,Random Forest,0.7,14,0.60,30,0.836


**Reading the table.** AUC isn't reported for the Week-4 rule because its score is a hard step function — most rows tie at exactly 0, so it isn't a genuine ranking over every page, only over the subset it flags. Precision@K sidesteps that: it only asks about the *top* of whichever ranking each method produces, which is the actually-actionable part of any of these for a reviewer with limited time.

If Random Forest doesn't clearly separate itself from Logistic Regression above — within a point or two at both K values — the honest read is to prefer Logistic Regression going forward: it's the one a non-engineer can be walked through, and per the training-honest-models principle, added complexity has to earn its place in this table, not be assumed.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
from sklearn.inspection import permutation_importance

# Permutation importance on Random Forest (the more flexible model) — scored on AUC since
# that's a standard, well-supported scorer; precision@50 is still the headline decision
# metric in the table above, this is specifically about which features the model leans on.
perm = permutation_importance(rf, X_test, y_test, scoring="roc_auc", n_repeats=15, random_state=42)
importance_table = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

print("What the model leans on (permutation importance, AUC-scored):")
print(importance_table)
top_feature = importance_table.iloc[0]["feature"]
print()
print(f"Top feature: {top_feature}")
print("Sanity check — does this make plausible sense, or is it suspiciously perfect (= likely leakage)?")
print("No feature here is a March-derived value, so a near-1.0 importance dominating everything")
print("else would be the leakage tell to watch for; a plausible, moderate lead is the expected shape.")

What the model leans on (permutation importance, AUC-scored):
                     feature  importance_mean  importance_std
0          days_since_update         0.183815        0.001982
1                    ctr_feb         0.039992        0.001644
2            impressions_feb         0.008722        0.000735
3           avg_position_feb         0.005853        0.000655
4               content_type         0.001951        0.000218
5               sessions_feb         0.000481        0.000451
6       avg_position_missing         0.000012        0.000011
7  days_since_update_missing         0.000000        0.000000

Top feature: days_since_update
Sanity check — does this make plausible sense, or is it suspiciously perfect (= likely leakage)?
No feature here is a March-derived value, so a near-1.0 importance dominating everything
else would be the leakage tell to watch for; a plausible, moderate lead is the expected shape.


In [7]:
# Concrete disagreement cases — where the model's top-ranked pages and the rule's
# flagged pages diverge, plus 3 wrong cases from the model to look at by hand.
test_df = test_df.copy()
test_df["model_score"] = rf.predict_proba(X_test)[:, 1]

model_top50_idx = np.argsort(-test_df["model_score"].values)[:50]
rule_flagged_idx = np.where(test_df["baseline_rule_score"].values > 0)[0]

model_only = set(model_top50_idx) - set(rule_flagged_idx)
rule_only = set(rule_flagged_idx) - set(model_top50_idx)

print(f"In the model's top 50 but the rule never flagged: {len(model_only)}")
print(f"Flagged by the rule but outside the model's top 50: {len(rule_only)}")
print()

if model_only:
    example = test_df.iloc[list(model_only)[0]]
    print("Example the model catches that the rule's hard AND-gates miss:")
    print(f"  days_since_update={example['days_since_update']:.0f} (rule needs >=180 exactly),"
          f" ctr_gap={example['ctr_gap']:.4f}, impressions_feb={example['impressions_feb']:.0f},"
          f" model_score={example['model_score']:.3f}")
    print("  -> a page just under the rule's hard staleness cutoff, or with a small-but-real CTR")
    print("     gap the AND-rule zeroes out, can still carry real signal a continuous model can use.")
print()

# 3 concrete wrong cases: highest-confidence model predictions that were actually wrong.
test_df["correct"] = (test_df["model_score"] >= 0.5).astype(int) == test_df["is_declining_next_month"]
wrong_confident = test_df[~test_df["correct"]].sort_values(
    "model_score", ascending=False, key=lambda s: np.abs(s - 0.5)
).head(3)

print("3 concrete wrong cases (model was confident, and wrong):")
for _, row in wrong_confident.iterrows():
    predicted = "decline" if row["model_score"] >= 0.5 else "stable"
    actual = "declined" if row["is_declining_next_month"] == 1 else "stayed stable"
    disappeared_note = " (page disappeared from March tracking entirely)" if row["disappeared_in_march"] else ""
    print(f"  predicted {predicted} (score={row['model_score']:.2f}), actually {actual}{disappeared_note}")
    print(f"    days_since_update={row['days_since_update']:.0f}, ctr_feb={row['ctr_feb']:.2%},"
          f" avg_position_feb={row['avg_position_feb']:.1f}, impressions_feb={row['impressions_feb']:.0f}")

In the model's top 50 but the rule never flagged: 50
Flagged by the rule but outside the model's top 50: 18

Example the model catches that the rule's hard AND-gates miss:
  days_since_update=3 (rule needs >=180 exactly), ctr_gap=0.0054, impressions_feb=32, model_score=0.827
  -> a page just under the rule's hard staleness cutoff, or with a small-but-real CTR
     gap the AND-rule zeroes out, can still carry real signal a continuous model can use.

3 concrete wrong cases (model was confident, and wrong):
  predicted stable (score=0.15), actually declined
    days_since_update=-114, ctr_feb=0.24%, avg_position_feb=19.2, impressions_feb=5881
  predicted decline (score=0.84), actually stayed stable
    days_since_update=3, ctr_feb=0.00%, avg_position_feb=10.5, impressions_feb=5
  predicted decline (score=0.84), actually stayed stable
    days_since_update=3, ctr_feb=0.00%, avg_position_feb=12.7, impressions_feb=5


**What the errors look like.** The clearest, most defensible pattern is the one the Week-4 rule structurally can't see: its hard AND-gates (staleness ≥ 180 days exactly, any CTR gap at all) zero out pages that are *almost* stale or have a small-but-real CTR gap, even when combined with strong visibility. A continuous model doesn't have that cliff — which is the concrete version of Week-3's finding that a hand-written AND-rule underperformed the base rate: the interactions between signals matter more than any single hard threshold.

The `disappeared_in_march` flag is worth reading separately from genuine content decline — a page that vanished from the panel entirely could reflect a real content problem, but it could just as easily reflect a client's tracking lapsing or churn, which no page-level feature here can distinguish. Any wrong case carrying that flag is a caution about the label's honesty, not just the model's.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.